In [1]:
import pandas as pd
import polars as pl
import time

In [3]:
DATA_PATH = "data/ucus_verisi.csv"

df = pd.read_csv(DATA_PATH)

df.tail()

,ucus_no,kalkis_havaalani,varis_havaalani,havayolu,ucus_tipi,kalkis_tarihi,bilet_fiyati,gecikme_dk
149995,F149996,TZX,SZF,Pegasus,dis_hat,2024-04-12,2049.93,49
149996,F149997,ADB,ESB,Pegasus,ic_hat,2023-01-12,442.01,19
149997,F149998,SAW,ADB,THY,ic_hat,2024-01-01,4084.65,42
149998,F149999,SAW,SAW,AnadoluJet,dis_hat,2023-11-29,2095.93,21
149999,F150000,TZX,ESB,AnadoluJet,dis_hat,2023-12-12,3628.82,4


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   ucus_no           150000 non-null  str    
 1   kalkis_havaalani  150000 non-null  str    
 2   varis_havaalani   150000 non-null  str    
 3   havayolu          150000 non-null  str    
 4   ucus_tipi         150000 non-null  str    
 5   kalkis_tarihi     150000 non-null  str    
 6   bilet_fiyati      150000 non-null  float64
 7   gecikme_dk        150000 non-null  int64  
dtypes: float64(1), int64(1), str(6)
memory usage: 9.2 MB


In [5]:
df.shape

(150000, 8)

In [6]:
df["kalkis_tarihi"] = pd.to_datetime(df["kalkis_tarihi"])

df.head()

,ucus_no,kalkis_havaalani,varis_havaalani,havayolu,ucus_tipi,kalkis_tarihi,bilet_fiyati,gecikme_dk
0,F000001,IST,IST,AnadoluJet,ic_hat,2023-04-13,1712.75,22
1,F000002,VAN,ADB,SunExpress,ic_hat,2024-03-11,1548.61,1
2,F000003,IST,SAW,Pegasus,ic_hat,2023-09-28,3619.77,34
3,F000004,IST,GZT,THY,dis_hat,2023-04-17,1776.45,2
4,F000005,IST,AYT,THY,ic_hat,2023-03-13,4736.62,3


In [14]:
def pandas_pipeline(dataframe):
    # Pipeline'ın çalışma süresini ölçmek için başlangıç zamanını alıyorum.
    start = time.perf_counter()

    # Veriyi önce gecikme ve bilet fiyatı koşullarına göre filtreliyorum.
    result = (
        dataframe[
            (dataframe["gecikme_dk"] > 30) &
            (dataframe["bilet_fiyati"] > 2000)
        ]

        # Filtrelenmiş veriyi havayolu ve uçuş tipi bazında grupluyorum.
        .groupby(["havayolu", "ucus_tipi"])

        # Her grup için ortalama fiyat, toplam gecikme ve uçuş sayısını hesaplıyorum.
        .agg(
            ortalama_fiyat=("bilet_fiyati", "mean"),
            toplam_gecikme=("gecikme_dk", "sum"),
            ucus_sayisi=("ucus_no", "count")
        )

        # Groupby sonucunu normal DataFrame formatına çeviriyorum.
        .reset_index()
    )

    # Pipeline bittikten sonra bitiş zamanını alıyorum.
    end = time.perf_counter()

    # Toplam çalışma süresini hesaplıyorum.
    elapsed_time = end - start

    # DataFrame'in bellek kullanımını MB cinsinden hesaplıyorum.
    memory_usage = dataframe.memory_usage(deep=True).sum() / (1024**2)

    # Sonuç DataFrame'i, süre ve bellek kullanımını döndürüyorum.
    return result, elapsed_time, memory_usage

In [15]:
# Pandas pipeline fonksiyonunu çalıştırıyorum.
# Fonksiyon bana:
# 1. İşlenmiş sonuç DataFrame'ini
# 2. Çalışma süresini
# 3. Bellek kullanımını döndürüyor.

pandas_result, pandas_time, pandas_memory = pandas_pipeline(df)

# Pipeline'ın toplam çalışma süresini saniye cinsinden yazdırıyorum.
print("Pandas Süre:", round(pandas_time, 4), "saniye")

# Pandas DataFrame'in yaklaşık bellek kullanımını MB cinsinden yazdırıyorum.
print("Pandas Bellek:", round(pandas_memory, 2), "MB")

# İşlenmiş sonuç DataFrame'inin ilk 5 satırını görüntülüyorum.
# Böylece aggregation işlemlerinin doğru çalışıp çalışmadığını kontrol edebiliyorum.
pandas_result.head()

Pandas Süre: 0.0342 saniye
Pandas Bellek: 47.86 MB


,havayolu,ucus_tipi,ortalama_fiyat,toplam_gecikme,ucus_sayisi
0,AnadoluJet,dis_hat,3174.021203,58608,1164
1,AnadoluJet,ic_hat,3179.906890,106210,2100
2,Corendon,dis_hat,3127.974025,20030,395
3,Corendon,ic_hat,3145.314519,35218,686
4,Pegasus,dis_hat,3158.403287,115374,2254


In [17]:
def polars_pipeline(data_path):

    # Pipeline'ın çalışma süresini ölçmek için başlangıç zamanını alıyorum.
    start = time.perf_counter()

    # Polars lazy pipeline oluşturuyorum.
    # scan_csv kullanarak veriyi lazy şekilde okuyorum.
    # Böylece veri hemen belleğe yüklenmiyor.
    result = (

        pl.scan_csv(data_path, try_parse_dates=True)

        # Veriyi filtreliyorum.
        # Sadece:
        # - gecikmesi 30 dakikadan fazla olan
        # - bilet fiyatı 2000'den büyük olan
        # uçuşları seçiyorum.
        .filter(
            (pl.col("gecikme_dk") > 30) &
            (pl.col("bilet_fiyati") > 2000)
        )

        # Veriyi havayolu ve uçuş tipine göre grupluyorum.
        .group_by(["havayolu", "ucus_tipi"])

        # Her grup için aggregation işlemleri uyguluyorum.
        .agg([

            # Ortalama bilet fiyatını hesaplıyorum.
            pl.col("bilet_fiyati")
            .mean()
            .alias("ortalama_fiyat"),

            # Toplam gecikme süresini hesaplıyorum.
            pl.col("gecikme_dk")
            .sum()
            .alias("toplam_gecikme"),

            # Her gruptaki uçuş sayısını hesaplıyorum.
            pl.col("ucus_no")
            .count()
            .alias("ucus_sayisi")
        ])

        # Lazy pipeline'ı çalıştırıp sonucu belleğe alıyorum.
        .collect()
    )

    # Pipeline bittikten sonra bitiş zamanını alıyorum.
    end = time.perf_counter()

    # Toplam çalışma süresini hesaplıyorum.
    elapsed_time = end - start

    # Sonuç DataFrame'inin yaklaşık bellek kullanımını MB cinsinden hesaplıyorum.
    memory_usage = result.estimated_size("mb")

    # Sonuç DataFrame'ini, süreyi ve bellek kullanımını döndürüyorum.
    return result, elapsed_time, memory_usage

In [18]:
# Polars pipeline fonksiyonunu çalıştırıyorum.
# Fonksiyon bana:
# 1. İşlenmiş sonuç DataFrame'ini
# 2. Çalışma süresini
# 3. Bellek kullanımını döndürüyor.

polars_result, polars_time, polars_memory = polars_pipeline(DATA_PATH)

# Polars pipeline'ın toplam çalışma süresini saniye cinsinden yazdırıyorum.
print("Polars Süre:", round(polars_time, 4), "saniye")

# Polars sonuç DataFrame'inin yaklaşık bellek kullanımını MB cinsinden yazdırıyorum.
print("Polars Bellek:", round(polars_memory, 2), "MB")

# İşlenmiş sonuç DataFrame'inin ilk 5 satırını görüntülüyorum.
# Böylece aggregation işlemlerinin doğru çalışıp çalışmadığını kontrol edebiliyorum.
polars_result.head()

Polars Süre: 0.0697 saniye
Polars Bellek: 0.0 MB


havayolu,ucus_tipi,ortalama_fiyat,toplam_gecikme,ucus_sayisi
str,str,f64,i64,u32
"""AnadoluJet""","""ic_hat""",3179.90689,106210,2100
"""SunExpress""","""dis_hat""",3223.253119,40542,776
"""THY""","""ic_hat""",3157.199388,286445,5674
"""Corendon""","""ic_hat""",3145.314519,35218,686
"""Corendon""","""dis_hat""",3127.974025,20030,395


In [19]:
# Pandas ve Polars benchmark sonuçlarını karşılaştırmalı bir tabloya aktarıyorum.
comparison_df = pd.DataFrame({

    # Kullanılan veri işleme kütüphanelerini belirtiyorum.
    "Kutuphane": ["Pandas", "Polars"],

    # Her kütüphanenin pipeline çalışma süresini saniye cinsinden ekliyorum.
    "Sure (sn)": [pandas_time, polars_time],

    # Her kütüphanenin yaklaşık bellek kullanımını MB cinsinden ekliyorum.
    "Bellek (MB)": [pandas_memory, polars_memory]
})

# Oluşturulan benchmark karşılaştırma tablosunu görüntülüyorum.
comparison_df

,Kutuphane,Sure (sn),Bellek (MB)
0,Pandas,0.034242,47.857380
1,Polars,0.069669,0.000325


In [20]:
# Farklı veri boyutlarında benchmark testi yapmak için
# satır sayılarını belirliyorum.
sizes = [10_000, 50_000, 100_000]

# Benchmark sonuçlarını saklamak için boş bir liste oluşturuyorum.
benchmark_results = []

# Her veri boyutu için pipeline testlerini tekrar çalıştırıyorum.
for size in sizes:

    # Ana veri setinden belirtilen satır sayısı kadar örnek veri alıyorum.
    sample_df = df.head(size)

    # Pandas pipeline'ını çalıştırıyorum.
    # Sonuç DataFrame'i, süre ve bellek kullanımını alıyorum.
    pandas_result, pandas_time, pandas_memory = pandas_pipeline(sample_df)

    # Polars pipeline'ını çalıştırıyorum.
    # Sonuç DataFrame'i, süre ve bellek kullanımını alıyorum.
    polars_result, polars_time, polars_memory = polars_pipeline(DATA_PATH)

    # Her test sonucunu sözlük yapısında benchmark listesine ekliyorum.
    benchmark_results.append({

        # Kullanılan veri boyutunu kaydediyorum.
        "satir_sayisi": size,

        # Pandas çalışma süresini saniye cinsinden kaydediyorum.
        "pandas_sure_sn": round(pandas_time, 4),

        # Polars çalışma süresini saniye cinsinden kaydediyorum.
        "polars_sure_sn": round(polars_time, 4),

        # Pandas bellek kullanımını MB cinsinden kaydediyorum.
        "pandas_bellek_mb": round(pandas_memory, 2),

        # Polars sonuç DataFrame'inin bellek kullanımını MB cinsinden kaydediyorum.
        "polars_sonuc_bellek_mb": round(polars_memory, 2)
    })

# Benchmark sonuç listesini DataFrame formatına dönüştürüyorum.
benchmark_df = pd.DataFrame(benchmark_results)

# Oluşturulan benchmark karşılaştırma tablosunu görüntülüyorum.
benchmark_df

,satir_sayisi,pandas_sure_sn,polars_sure_sn,pandas_bellek_mb,polars_sonuc_bellek_mb
0,10000,0.0156,0.0662,3.19,0.0
1,50000,0.0214,0.0431,15.95,0.0
2,100000,0.0214,0.0416,31.91,0.0


In [21]:
# Pandas ve Polars çalışma sürelerini karşılaştırmak için
# hız farkı oranını hesaplıyorum.

benchmark_df["hiz_farki"] = (

    # Pandas çalışma süresini
    benchmark_df["pandas_sure_sn"]

    # Polars çalışma süresine bölüyorum.
    / benchmark_df["polars_sure_sn"]

# Sonucu daha okunabilir olması için 2 basamağa yuvarlıyorum.
).round(2)

# Güncellenmiş benchmark tablosunu görüntülüyorum.
benchmark_df

,satir_sayisi,pandas_sure_sn,polars_sure_sn,pandas_bellek_mb,polars_sonuc_bellek_mb,hiz_farki
0,10000,0.0156,0.0662,3.19,0.0,0.24
1,50000,0.0214,0.0431,15.95,0.0,0.50
2,100000,0.0214,0.0416,31.91,0.0,0.51
